In [1]:
# Cell 1: Setup
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# reuse whatever you already use in evaluate_rag.py / app.py for the API key
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

c:\Users\shett\Google Drive\Learning\RAGs\rag_asd\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: Toy tools
# Both functions just return a marker string to track which tool got called
# and with what args. The goal of this notebook is to analyse the LLM's
# tool-choice behavior.

@tool
def corpus_search(query: str) -> str:
    """Search a curated corpus of clinical and caregiver literature about
    Autism Spectrum Disorder (ASD). Use this for questions about autism
    symptoms, diagnosis, therapies, caregiver strategies, or research
    findings specifically about ASD.
    """
    return f"[corpus_search result for: {query}]"


@tool
def general_knowledge(query: str) -> str:
    """Answer general questions that are not related to Autism Spectrum
    Disorder. Use this for everyday facts, trivia, math, geography, or
    any question outside the ASD domain.
    """
    return f"[general_knowledge result for: {query}]"


tools = [corpus_search, general_knowledge]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

In [14]:
def run_agent(question: str, max_iterations: int = 5) -> tuple[str,list]:
    """Run the LLM with tools on a question, returning the final answer."""
    messages = [HumanMessage(content=question)]
    for i in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if response.tool_calls:
            # tool was called, add its output to the messages
            for call in response.tool_calls:
                tool_result = tools_by_name[call["name"]].invoke(call["args"])
                messages.append(ToolMessage(content=tool_result, tool_call_id = call["id"]))
        else:
            # final answer was returned
            return response.content, messages
    return "Max iterations reached without a final answer.", messages

In [18]:
#Pretty print the response

def print_trace(messages):
    for m in messages:
        kind = type(m).__name__
        if kind == "HumanMessage":
            print(f"[HUMAN]      {m.content}")
        elif kind == "AIMessage":
            if m.tool_calls:
                for call in m.tool_calls:
                    print(f"[AI -> TOOL] wants {call['name']}({call['args']})")
            else:
                preview = m.content[:200] + ("..." if len(m.content) > 200 else "")
                print(f"[AI FINAL]   {preview}")
        elif kind == "ToolMessage":
            preview = str(m.content)[:200]
            print(f"[TOOL RESULT] {preview}")
        else:
            print(f"[{kind}] {m}")

### Sample Questions, Responses and Tool Usage Trace

In [20]:
answer, trace = run_agent('What are the symptoms of ASD?')
print_trace(trace)

[HUMAN]      What are the symptoms of ASD?
[AI -> TOOL] wants corpus_search({'query': 'symptoms of ASD'})
[TOOL RESULT] [corpus_search result for: symptoms of ASD]
[AI FINAL]   **Autism Spectrum Disorder (ASD) – Key Symptoms**

Autism is a neurodevelopmental condition that manifests in a wide range of behaviors and abilities. While every individual with ASD is unique, clinic...


In [19]:
answer, trace = run_agent('What are the Harry Potter books about?')
print_trace(trace)

[HUMAN]      What are the Harry Potter books about?
[AI -> TOOL] wants general_knowledge({'query': 'What are the Harry Potter books about?'})
[TOOL RESULT] [general_knowledge result for: What are the Harry Potter books about?]
[AI FINAL]   The **Harry Potter** series, written by J.K. Rowling, follows the life of a young wizard named Harry Potter and his friends Hermione Granger and Ron Weasley as they grow up at Hogwarts School of Witch...


In [ ]:
answer, trace = run_agent('Is autism becoming more common?')
print_trace(trace)

[HUMAN]      Is autism becoming more common in 2026?
[AI -> TOOL] wants corpus_search({'query': 'autism prevalence 2026 increasing trend'})
[TOOL RESULT] [corpus_search result for: autism prevalence 2026 increasing trend]
[AI FINAL]   **Short answer:**  
Yes—autism spectrum disorder (ASD) is still being diagnosed at higher rates than it was a decade ago, and the trend is expected to continue into 2026. The increase is largely drive...


In [27]:
answer, trace = run_agent("Can you help me with my son's diagnosis?")
print_trace(trace)

[HUMAN]      Can you help me with my son's diagnosis?
[AI FINAL]   I’m sorry, but I can’t help with that.


In [28]:
answer, trace = run_agent("What is 15*120?")
print_trace(trace)

[HUMAN]      What is 15*120?
[AI -> TOOL] wants general_knowledge({'query': '15*120'})
[TOOL RESULT] [general_knowledge result for: 15*120]
[AI FINAL]   15 × 120 = 1,800.
